In [1]:
from pathlib import Path
import pandas as pd

Step 1: Load data into dataframes for analysis and review

In [ ]:
data_dir = Path("../datos")

etiquetas = pd.read_csv(data_dir / "etiquetas_train.csv")
combos = pd.read_csv(data_dir / "combos_a_predecir.csv")

facturas_encabezado         = pd.read_parquet(data_dir / "oinv.parquet")
facturas_detalle            = pd.read_parquet(data_dir / "inv1.parquet")
notas_credito_encabezado    = pd.read_parquet(data_dir / "orin.parquet")
notas_credito_detalle       = pd.read_parquet(data_dir / "rin1.parquet")
catalogo_productos          = pd.read_parquet(data_dir / "oitm.parquet")
grupo_productos             = pd.read_parquet(data_dir / "oitb.parquet")
inventario_diario           = pd.read_parquet(data_dir / "inventario_regional.parquet")
ubicaciones                 = pd.read_parquet(data_dir / "obin.parquet")
bodegas                     = pd.read_parquet(data_dir / "owhs.parquet")

In [3]:
facturas_encabezado.info()
facturas_encabezado.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7206 entries, 0 to 7205
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   DocEntry    7206 non-null   int64         
 1   DocDate     7206 non-null   datetime64[ms]
 2   Canceled    7206 non-null   object        
 3   DocSubType  7206 non-null   object        
 4   DocCur      7206 non-null   object        
 5   DocRate     7206 non-null   float64       
dtypes: datetime64[ms](1), float64(1), int64(1), object(3)
memory usage: 337.9+ KB


,DocEntry,DocDate,Canceled,DocSubType,DocCur,DocRate
0,555222,2026-08-01,N,--,QTZ,1.0
1,555223,2026-08-01,N,--,QTZ,1.0
2,555224,2026-08-01,N,--,QTZ,1.0
3,555225,2026-08-01,N,--,QTZ,1.0
4,555226,2026-08-01,N,--,QTZ,1.0


In [4]:
facturas_detalle.info()
facturas_detalle.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18386 entries, 0 to 18385
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DocEntry     18386 non-null  int64  
 1   LineNum      18386 non-null  int64  
 2   ItemCode     18378 non-null  object 
 3   WhsCode      18378 non-null  object 
 4   U_Ubicacion  16855 non-null  float64
 5   Quantity     18386 non-null  float64
 6   LineTotal    18386 non-null  float64
 7   StockValue   18386 non-null  float64
 8   GrssProfit   18386 non-null  float64
dtypes: float64(5), int64(2), object(2)
memory usage: 1.3+ MB


,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit
0,555222,0,4861,GT1CEDI,111.0,1.0,4463.39,3952.26,511.13
1,555223,0,3916,GT1COMER,50.0,1.0,891.96,852.44,39.52
2,555224,0,4617,GT1COMER,53.0,1.0,7141.96,6798.29,343.67
3,555225,0,4549,GT1COMER,50.0,1.0,150.89,102.66,48.23
4,555225,1,5947,GT1COMER,50.0,1.0,106.25,68.10,38.15
5,555225,2,2992,GT1COMER,50.0,1.0,445.54,238.97,206.57
6,555225,3,2472,GT1COMER,50.0,1.0,1427.68,1900.38,-472.70
7,555226,0,4183,GT1COMER,50.0,1.0,891.96,878.09,13.87
8,555227,0,6382,GT1COMER,33.0,1.0,107.14,0.00,107.14
9,555228,0,5754,GT1COMER,53.0,1.0,2677.68,2678.28,-0.60


oinv.parquet    Facturas – encabezado
                (DocEntry, DocDate, Canceled, DocSubType, DocCur, DocRate). DocSubType 'DN' = nota de débito.
inv1.parquet    Facturas – detalle
                (DocEntry, LineNum, ItemCode, WhsCode,U_Ubicacion, Quantity, LineTotal, StockValue = costo, GrssProfit)

facturas_encabezado.DocEntry(one) - facturas_detalle.DocEntry(many)


In [5]:
facturas_encabezado.duplicated(subset=["DocEntry"]).sum()

np.int64(0)

In [6]:
facturas_detalle.duplicated(subset=["DocEntry", "LineNum"]).sum()

np.int64(0)

In [7]:
facturas_encabezado["DocDate"].agg(["min","max"])

min   2026-08-01
max   2026-08-21
Name: DocDate, dtype: datetime64[ns]

In [8]:
facturas_encabezado[["DocCur","DocRate"]].value_counts(dropna=False)

DocCur  DocRate
QTZ     1.00       7086
USD     7.66        120
Name: count, dtype: int64

In [9]:
facturas_encabezado["Canceled"].value_counts(dropna=False)

Canceled
N    7156
Y      25
C      25
Name: count, dtype: int64

In [10]:
facturas_encabezado["DocSubType"].value_counts(dropna=False)

DocSubType
--    7206
Name: count, dtype: int64

In [11]:
facturas_detalle.isna().sum()

DocEntry          0
LineNum           0
ItemCode          8
WhsCode           8
U_Ubicacion    1531
Quantity          0
LineTotal         0
StockValue        0
GrssProfit        0
dtype: int64

In [12]:
facturas_detalle[["ItemCode","WhsCode","U_Ubicacion"]].isna().value_counts()

ItemCode  WhsCode  U_Ubicacion
False     False    False          16855
                   True            1523
True      True     True               8
Name: count, dtype: int64

Identified some missing values for facturas_detalle on ItemCode, WhsCode, U_Ubicacion

In [13]:
missing_reference = facturas_detalle[["ItemCode","WhsCode","U_Ubicacion"]].isna().any(axis=1)
facturas_detalle.loc[missing_reference].head(10)

,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit
1537,556190,1,3903-001003,GT1SERV,NaN,1.0,0.00,59.37,-59.37
1538,556190,2,SC-001,GT1SERV,NaN,1.0,312.50,0.00,312.50
1814,556341,0,SC-001,GT1SERV,NaN,1.0,3013.39,0.00,3013.39
1815,556341,1,5735,GT1SERV,NaN,1.0,0.00,92.86,-92.86
1816,556341,2,SC-559,GT1SERV,NaN,8.0,0.00,2.93,-2.93
1817,556342,0,SC-001,GT1SERV,NaN,1.0,580.36,0.00,580.36
2197,556486,0,GH82-33392A,GT1SERV,NaN,1.0,0.00,32.76,-32.76
2198,556486,1,GH82-33544A,GT1SERV,NaN,1.0,0.00,41.21,-41.21
2199,556486,2,GH82-33385A,GT1SERV,NaN,1.0,0.00,1139.53,-1139.53
2200,556486,3,SC-001,GT1SERV,NaN,1.0,1964.29,0.00,1964.29


In [14]:
facturas_detalle.loc[facturas_detalle["ItemCode"].isna()]

,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit
3673,556934,0,None,None,NaN,0.0,4887.50,0.0,0.0
5934,557473,0,None,None,NaN,0.0,1191.89,0.0,0.0
5935,557473,1,None,None,NaN,0.0,12523.97,0.0,0.0
12155,559797,0,None,None,NaN,0.0,8708.93,0.0,0.0
12158,559799,0,None,None,NaN,0.0,35398.34,0.0,0.0
12162,559803,0,None,None,NaN,0.0,2041.92,0.0,0.0
15480,561243,0,None,None,NaN,0.0,86035.31,0.0,0.0
16915,561743,0,None,None,NaN,0.0,16462.50,0.0,0.0


In [15]:
missing_header = ~facturas_detalle["DocEntry"].isin(facturas_encabezado["DocEntry"])
missing_header.sum()

np.int64(0)

Every invoice line references an existing header Header keys and document line keys are unique and complete. Investigation into missing data revealed that 8 rows are the same that are missing ItemCode, WhsCode and U_Ubicacion leaving 1523 rows with only missing U_Ubicacion.

In [16]:
missing_item_lines = facturas_detalle.loc[facturas_detalle["ItemCode"].isna()]
missing_item_lines.merge(facturas_encabezado, on="DocEntry", how="left", validate="many_to_one")

,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit,DocDate,Canceled,DocSubType,DocCur,DocRate
0,556934,0,None,None,NaN,0.0,4887.50,0.0,0.0,2026-08-05,N,--,QTZ,1.0
1,557473,0,None,None,NaN,0.0,1191.89,0.0,0.0,2026-08-07,N,--,QTZ,1.0
2,557473,1,None,None,NaN,0.0,12523.97,0.0,0.0,2026-08-07,N,--,QTZ,1.0
3,559797,0,None,None,NaN,0.0,8708.93,0.0,0.0,2026-08-14,N,--,QTZ,1.0
4,559799,0,None,None,NaN,0.0,35398.34,0.0,0.0,2026-08-14,N,--,QTZ,1.0
5,559803,0,None,None,NaN,0.0,2041.92,0.0,0.0,2026-08-14,N,--,QTZ,1.0
6,561243,0,None,None,NaN,0.0,86035.31,0.0,0.0,2026-08-17,N,--,QTZ,1.0
7,561743,0,None,None,NaN,0.0,16462.50,0.0,0.0,2026-08-19,N,--,QTZ,1.0


In [17]:
facturas_detalle.loc[facturas_detalle["U_Ubicacion"].isna(),"WhsCode"].value_counts(dropna=False)

WhsCode
GT1SERV    1482
01           23
GT2SERV      10
None          8
GT3SERV       8
Name: count, dtype: int64

In [18]:
facturas_detalle[["WhsCode"]].value_counts(dropna=False)

WhsCode 
GT1COMER    12402
GT1CEDI      3266
GT1SERV      1482
GT2CEDI      1048
GT3COMER      106
GT1B2B         29
01             23
GT2SERV        10
GT3SERV         8
NaN             8
GT3CEDI         4
Name: count, dtype: int64

In [19]:
whcodes = ["GT1SERV","01","GT2SERV"]
bodegas.loc[bodegas["WhsCode"].isin(whcodes)]

,WhsCode,WhsName,Canal
0,01,Almacén general,RETAIL
17,GT1SERV,SERVICE CENTER,LOGISTICA
22,GT2SERV,SERVICE CENTER VIRTUAL,LOGISTICA


In [20]:
ubicaciones.loc[ubicaciones["WhsCode"].isin(whcodes)]

,AbsEntry,Ubicacion,WhsCode,Canal
7,8,GT1SERV-SYSTEM-BIN-LOCATION,GT1SERV,None
14,15,GT2SERV-UBICACIÓN-DE-SISTEMA,GT2SERV,None
74,76,GT1SERV-GT-001-UBI-SERVIFIS-PRODUCTO,GT1SERV,SERVICE
75,77,GT1SERV-GT-001-UBI-SERVIFIS-REPUESTO,GT1SERV,SERVICE
76,78,GT1SERV-GT-001-UBI-SERVIFIS-TRADEIN,GT1SERV,SERVICE
81,83,GT2SERV-GT-001-UBI-SERVIVIRT-RMA,GT2SERV,SERVICE
82,84,GT2SERV-GT-002-UBI-SERVIVIRT-TRADEIN,GT2SERV,SERVICE
116,121,GT2SERV-GT-001-UBI-SERVIVIRT-ODS,GT2SERV,SERVICE


Based on above data, Almacen General(01), GT1SERV and GT2SERV are missing U_Ubicacion data, since this is just a location identifier linked to ubicaciones.AbsEntry, these lines will be kept for the investigation.

In [21]:
notas_credito_encabezado.info()
notas_credito_encabezado.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   DocEntry  224 non-null    int64         
 1   DocDate   224 non-null    datetime64[ms]
 2   Canceled  224 non-null    object        
dtypes: datetime64[ms](1), int64(1), object(1)
memory usage: 5.4+ KB


,DocEntry,DocDate,Canceled
0,20505,2026-08-03,N
1,20508,2026-08-03,N
2,20510,2026-08-03,N
3,20511,2026-08-03,N
4,20512,2026-08-03,N


In [22]:
notas_credito_detalle.info()
notas_credito_detalle.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 431 entries, 0 to 430
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DocEntry     431 non-null    int64  
 1   LineNum      431 non-null    int64  
 2   ItemCode     381 non-null    object 
 3   WhsCode      381 non-null    object 
 4   U_Ubicacion  324 non-null    float64
 5   Quantity     431 non-null    float64
 6   LineTotal    431 non-null    float64
 7   StockValue   431 non-null    float64
 8   GrssProfit   431 non-null    float64
 9   BaseEntry    431 non-null    int64  
 10  BaseType     431 non-null    int64  
dtypes: float64(5), int64(4), object(2)
memory usage: 37.2+ KB


,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit,BaseEntry,BaseType
0,20505,0,5145,GT1CEDI,18.0,1.0,383.00,266.45,116.55,556305,13
1,20508,0,GH81-23901A,GT1SERV,NaN,1.0,0.00,46.53,-46.53,548785,13
2,20508,1,GH82-31952A,GT1SERV,NaN,1.0,0.00,85.45,-85.45,549202,13
3,20508,2,SC-001,GT1SERV,NaN,1.0,267.86,0.00,267.86,556190,13
4,20510,0,3301,GT1CEDI,20.0,2.0,5954.43,5955.13,-0.70,555981,13


orin.parquet        Notas de crédito – encabezado
                    (DocEntry, DocDate, Canceled)
                    
rin1.parquet        Notas de crédito – detalle
                    (mismas columnas que facturas_detalle + BaseEntry = factura que origina la devolución)

In [23]:
notas_credito_encabezado.duplicated(subset=["DocEntry"]).sum()

np.int64(0)

In [24]:
notas_credito_detalle.duplicated(subset=["DocEntry","LineNum"]).sum()

np.int64(0)

In [25]:
notas_credito_detalle.isna().sum()

DocEntry         0
LineNum          0
ItemCode        50
WhsCode         50
U_Ubicacion    107
Quantity         0
LineTotal        0
StockValue       0
GrssProfit       0
BaseEntry        0
BaseType         0
dtype: int64

In [26]:
notas_credito_detalle[["ItemCode","WhsCode","U_Ubicacion"]].isna().value_counts()

ItemCode  WhsCode  U_Ubicacion
False     False    False          324
                   True            57
True      True     True            50
Name: count, dtype: int64

In [27]:
notas_credito_detalle.loc[notas_credito_detalle["ItemCode"].isna()]

,DocEntry,LineNum,ItemCode,WhsCode,U_Ubicacion,Quantity,LineTotal,StockValue,GrssProfit,BaseEntry,BaseType
25,20524,0,None,None,NaN,1.0,2391.90,0.0,0.0,551900,13
31,20527,0,None,None,NaN,1.0,5715.21,0.0,0.0,553665,13
32,20528,0,None,None,NaN,1.0,23851.54,0.0,0.0,548805,13
50,20537,0,None,None,NaN,0.0,26785.71,0.0,0.0,550337,13
68,20549,0,None,None,NaN,0.0,54933.04,0.0,0.0,552176,13
146,20590,0,None,None,NaN,1.0,28142.92,0.0,0.0,554118,13
147,20591,0,None,None,NaN,1.0,9513.67,0.0,0.0,548835,13
148,20592,0,None,None,NaN,1.0,1239.95,0.0,0.0,553558,13
149,20593,0,None,None,NaN,1.0,3150.54,0.0,0.0,551090,13
150,20594,0,None,None,NaN,1.0,7992.63,0.0,0.0,548662,13


In [28]:
notas_credito_detalle.loc[notas_credito_detalle["U_Ubicacion"].isna(),"WhsCode"].value_counts(dropna=False)

WhsCode
None        50
GT1COMER    20
GT1CEDI     20
01          12
GT1SERV      3
GT2CEDI      2
Name: count, dtype: int64

In [29]:
missing_header2 = ~notas_credito_detalle["DocEntry"].isin(notas_credito_encabezado["DocEntry"])
missing_header2.sum()

np.int64(0)

In [30]:
notas_credito_encabezado["DocDate"].agg(["min","max"])

min   2026-08-03
max   2026-08-21
Name: DocDate, dtype: datetime64[ns]

In [31]:
notas_credito_encabezado["Canceled"].value_counts(dropna=False)

Canceled
N    224
Name: count, dtype: int64

In [32]:
notas_credito_detalle["BaseType"].value_counts(dropna=False)

BaseType
13    431
Name: count, dtype: int64

In [33]:
missing_base = ~notas_credito_detalle["BaseEntry"].isin(facturas_encabezado["DocEntry"])
notas_credito_detalle.loc[missing_base,["DocEntry","LineNum","BaseType","BaseEntry","LineTotal"]]

,DocEntry,LineNum,BaseType,BaseEntry,LineTotal
1,20508,0,13,548785,0.00
2,20508,1,13,549202,0.00
6,20512,0,13,549879,9103.57
25,20524,0,13,551900,2391.90
31,20527,0,13,553665,5715.21
...,...,...,...,...,...
388,20727,0,13,552872,237.79
389,20728,0,13,548371,1643.67
416,20752,0,13,552852,338868.66
428,20763,2,13,549718,16070.54


Identified 70 credit-note lines with unmatched invoice references from the table facturas_encabezado

notas_credito_encabezado contains 224 credit-note headers and notas_credito_detalle contains 431 lines.
Their keys are unique and complete, and every line matches a header.
All headers have Canceled = N, with dates from August 3 to August 21.

50 lines are missing ItemCode, WhsCode and U_Ubicacion together.
Another 57 are missing only U_Ubicacion. These lines remain unchanged
while their treatment is investigated.

In [34]:
grupo_productos.info()
grupo_productos.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ItmsGrpCod     12 non-null     int64 
 1   GrupoArticulo  12 non-null     object
dtypes: int64(1), object(1)
memory usage: 324.0+ bytes


,ItmsGrpCod,GrupoArticulo
0,100,Artículos
1,165,AUDIO Y VIDEO
2,174,COMPUTACION Y ACCESORIOS
3,173,ELECTRODOMESTICOS PEQUEÑOS
4,168,GAMING


In [35]:
grupo_productos.isna().sum()

ItmsGrpCod       0
GrupoArticulo    0
dtype: int64

In [36]:
grupo_productos.duplicated(["ItmsGrpCod"]).sum()

np.int64(0)

In [37]:
catalogo_productos.info()
catalogo_productos.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12847 entries, 0 to 12846
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ItemCode      12847 non-null  object
 1   InvntryUom    11580 non-null  object
 2   ItmsGrpCod    12847 non-null  int64 
 3   U_Familia     9427 non-null   object
 4   U_SubFamilia  9427 non-null   object
 5   U_Segmento    9375 non-null   object
dtypes: int64(1), object(5)
memory usage: 602.3+ KB


,ItemCode,InvntryUom,ItmsGrpCod,U_Familia,U_SubFamilia,U_Segmento
0,0110,MI-KXKMSWWC210,164,ACCESORIOS,SOFTWARE,KNOX
1,0111,MI-KXKSSWWC210,164,ACCESORIOS,SOFTWARE,KNOX
2,0112,SM-T575NZKLGTO,164,TABLETS,ACTIVEX,LTE
3,0113,SM-T636BZKLGTO,164,TABLETS,ACTIVEX,LTE
4,0114,DF10A9500CG/A1,166,ARMARIO,N/A,N/A


In [38]:
catalogo_productos.isna().sum()

ItemCode           0
InvntryUom      1267
ItmsGrpCod         0
U_Familia       3420
U_SubFamilia    3420
U_Segmento      3472
dtype: int64

In [39]:
catalogo_productos.duplicated(["ItemCode"]).sum()

np.int64(10)

In [40]:
catalogo_productos.loc[catalogo_productos.duplicated(["ItemCode"], keep=False)].sort_values("ItemCode")

,ItemCode,InvntryUom,ItmsGrpCod,U_Familia,U_SubFamilia,U_Segmento
3426,4371,SM-A065MLGACGU,164,CELULARES,SERIE A,A0X
12837,4371,SM-A065MLGACGU,164,CELULARES,SERIE A,SIN SEGMENTO
3807,4722,SM-A165MZAACGU,164,CELULARES,SERIE A,A1X
12838,4722,SM-A165MZAACGU,164,CELULARES,SERIE A,SIN SEGMENTO
4688,5654,SM-A075MZKECGU,164,CELULARES,SERIE A,A0X
12839,5654,SM-A075MZKECGU,164,CELULARES,SERIE A,SIN SEGMENTO
4690,5656,SM-A175FZKICGU,164,CELULARES,SERIE A,A1X
12840,5656,SM-A175FZKICGU,164,CELULARES,SERIE A,SIN SEGMENTO
5104,6065,SM-A175FLBICGU,164,CELULARES,SERIE A,A1X
12841,6065,SM-A175FLBICGU,164,CELULARES,SERIE A,SIN SEGMENTO


For these duplicate values, we will proceed with dropping the "SIN SEGMENTO" rows since all other cols match to avoid possible data inflation due to duplicate item codes.

In [41]:
catalogo_productos.loc[~catalogo_productos["ItmsGrpCod"].isin(grupo_productos["ItmsGrpCod"])]

,ItemCode,InvntryUom,ItmsGrpCod,U_Familia,U_SubFamilia,U_Segmento


In [42]:
grupo_productos[grupo_productos["ItmsGrpCod"]==164]

,ItmsGrpCod,GrupoArticulo
11,164,TELEFONIA Y ACC


grupo_productos contains 12 groups with unique keys and no missing values.
Every ItmsGrpCod in catalogo_productos matches an existing group.

catalogo_productos has 1,267 rows without a SKU in InvntryUom. These remain
unmapped rather than being assigned ItemCode as a replacement.

In [43]:
bodegas.info()
bodegas.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   WhsCode  44 non-null     object
 1   WhsName  44 non-null     object
 2   Canal    44 non-null     object
dtypes: object(3)
memory usage: 1.2+ KB


,WhsCode,WhsName,Canal
0,01,Almacén general,RETAIL
1,ACT-EPP,EPP,RETAIL
2,B2B,B2B,B2B
3,B2B-IMPO,B2B Importaciones,OTROS
4,BED-CD,Empaque en mal estado (apto para la venta),OFICINA


In [44]:
bodegas.duplicated(["WhsCode"]).sum()

np.int64(0)

In [45]:
bodegas.isna().sum()

WhsCode    0
WhsName    0
Canal      0
dtype: int64

In [46]:
ubicaciones.info()
ubicaciones.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   AbsEntry   135 non-null    int64 
 1   Ubicacion  135 non-null    object
 2   WhsCode    135 non-null    object
 3   Canal      116 non-null    object
dtypes: int64(1), object(3)
memory usage: 4.3+ KB


,AbsEntry,Ubicacion,WhsCode,Canal
0,1,WS - NAR-UBICACIÓN-DE-SISTEMA,WS - NAR,None
1,2,GT1B2B-SYSTEM-BIN-LOCATION,GT1B2B,None
2,3,GT1CEDI-SYSTEM-BIN-LOCATION,GT1CEDI,None
3,4,GT1COMER-SYSTEM-BIN-LOCATION,GT1COMER,None
4,5,GT1CONTI-SYSTEM-BIN-LOCATION,GT1CONTI,None


In [47]:
ubicaciones.duplicated(["AbsEntry"]).sum()

np.int64(0)

In [48]:
ubicaciones.isna().sum()

AbsEntry      0
Ubicacion     0
WhsCode       0
Canal        19
dtype: int64

In [49]:
ubicaciones.loc[~ubicaciones["WhsCode"].isin(bodegas["WhsCode"])]

,AbsEntry,Ubicacion,WhsCode,Canal
0,1,WS - NAR-UBICACIÓN-DE-SISTEMA,WS - NAR,None
91,93,GT5IMPO-UBICACIÓN-DE-SISTEMA,GT5IMPO,None
92,96,GT5IMPO-GT-005-UBI-TRANS5-TIGO,GT5IMPO,DISTY


In [50]:
ubicaciones.loc[ubicaciones["Canal"].isna()]

,AbsEntry,Ubicacion,WhsCode,Canal
0,1,WS - NAR-UBICACIÓN-DE-SISTEMA,WS - NAR,None
1,2,GT1B2B-SYSTEM-BIN-LOCATION,GT1B2B,None
2,3,GT1CEDI-SYSTEM-BIN-LOCATION,GT1CEDI,None
3,4,GT1COMER-SYSTEM-BIN-LOCATION,GT1COMER,None
4,5,GT1CONTI-SYSTEM-BIN-LOCATION,GT1CONTI,None
5,6,GT1DAÑO-SYSTEM-BIN-LOCATION,GT1DAÑO,None
6,7,GT1RECL-SYSTEM-BIN-LOCATION,GT1RECL,None
7,8,GT1SERV-SYSTEM-BIN-LOCATION,GT1SERV,None
8,9,GT2CEDI-SYSTEM-BIN-LOCATION,GT2CEDI,None
9,10,GT2COMER-SYSTEM-BIN-LOCATION,GT2COMER,None


bodegas contains 44 warehouses with no duplicate keys or missing values. ubicaciones contains 135 locations with no duplicate AbsEntry keys. Three locations reference warehouse codes absent from bodegas: WS - NAR and GT5IMPO.

Identified 19 locations with missing Canal, all named as system locations. These channels will remain unknown rather than being replaced with the warehouse channel.

In [51]:
for name, lines in [('facturas_detalle', facturas_detalle), ('notas_credito_detalle', notas_credito_detalle)]:
    unmatched = (
        lines["U_Ubicacion"].notna()
        & ~lines["U_Ubicacion"].isin(ubicaciones["AbsEntry"])
    )
    print(name, "unmatched location references:", unmatched.sum())

facturas_detalle unmatched location references: 0
notas_credito_detalle unmatched location references: 0


In [52]:
for name, lines in [('facturas_detalle', facturas_detalle), ('notas_credito_detalle', notas_credito_detalle)]:
    for column, catalog in [
        ("ItemCode", catalogo_productos["ItemCode"]),
        ("WhsCode", bodegas["WhsCode"]),
    ]:
        unmatched = lines[column].notna() & ~lines[column].isin(catalog)
        print(name, column, "unmatched:", unmatched.sum())

facturas_detalle ItemCode unmatched: 0
facturas_detalle WhsCode unmatched: 0
notas_credito_detalle ItemCode unmatched: 0
notas_credito_detalle WhsCode unmatched: 0


All nonmissing U_Ubicacion values in facturas_detalle and notas_credito_detalle match an existing ubicaciones.AbsEntry. Checking whether each line's warehouse agrees with its location's warehouse remains pending.

In [53]:
inventario_diario.info()
inventario_diario.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502838 entries, 0 to 502837
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   DataMartDate       502838 non-null  datetime64[ns]
 1   pais               502838 non-null  object        
 2   Canal              501492 non-null  object        
 3   Ubicacion          502838 non-null  object        
 4   CodigoAlmacen      502838 non-null  object        
 5   GrupoArticulo      502838 non-null  object        
 6   SKU                502749 non-null  object        
 7   sum_Stock          502838 non-null  int64         
 8   sum_Comprometido   502838 non-null  int64         
 9   sum_Pedido         502838 non-null  int64         
 10  sum_CostoTotal     502391 non-null  float64       
 11  sum_CostoTotalUSD  502391 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(6)
memory usage: 46.0+ MB


,DataMartDate,pais,Canal,Ubicacion,CodigoAlmacen,GrupoArticulo,SKU,sum_Stock,sum_Comprometido,sum_Pedido,sum_CostoTotal,sum_CostoTotalUSD
0,2026-08-01,SLC-GT,,GT5CEDI-GT-001-UBI-NC-MAYOR,GT5CEDI,TELEFONIA Y ACC,SM-A065MLBACGU,24,0,0,12850.039992,1685.29
1,2026-08-01,SLC-GT,,GT5CEDI-GT-001-UBI-NC-MAYOR,GT5CEDI,TELEFONIA Y ACC,SM-A065MLGACGU,14,0,0,8763.409998,1149.32
2,2026-08-01,SLC-GT,,GT5CEDI-GT-001-UBI-NC-MAYOR,GT5CEDI,TELEFONIA Y ACC,SM-A065MZKACGU,18,0,0,9630.308574,1263.02
3,2026-08-01,SLC-GT,,GT5CEDI-GT-001-UBI-NC-MAYOR,GT5CEDI,TELEFONIA Y ACC,SM-A165MZAACGU,1,0,0,800.670000,105.01
4,2026-08-01,SLC-GT,,GT5CEDI-GT-003-UBI-FIS3-TIGO,GT5CEDI,TELEFONIA Y ACC,SM-A165MZKACGU,1,0,0,800.610000,105.00


In [54]:
inventario_diario.isna().sum()

DataMartDate            0
pais                    0
Canal                1346
Ubicacion               0
CodigoAlmacen           0
GrupoArticulo           0
SKU                    89
sum_Stock               0
sum_Comprometido        0
sum_Pedido              0
sum_CostoTotal        447
sum_CostoTotalUSD     447
dtype: int64

In [55]:
text_columns = inventario_diario.select_dtypes(include="object").columns
inventario_diario[text_columns].apply(lambda column: column.str.strip().eq("").sum())

pais              0
Canal            30
Ubicacion         0
CodigoAlmacen     0
GrupoArticulo     0
SKU               0
dtype: int64

inventario_diario contains 502,838 rows. Identified 1,346 missing Canal values and another 30 blank or whitespace-only channels, 89 missing SKUs, and 447 missing values in each cost column. Their overlap and treatment remain to be investigated.

In [56]:
inventario_diario["DataMartDate"].agg(["min","max","nunique"])

min        2026-08-01 00:00:00
max        2026-08-21 00:00:00
nunique                     21
Name: DataMartDate, dtype: object

In [57]:
inventory_coverage = pd.crosstab(inventario_diario["CodigoAlmacen"], inventario_diario["DataMartDate"])
inventory_coverage

DataMartDate,2026-08-01,2026-08-02,2026-08-03,2026-08-04,2026-08-05,2026-08-06,2026-08-07,2026-08-08,2026-08-09,2026-08-10,...,2026-08-12,2026-08-13,2026-08-14,2026-08-15,2026-08-16,2026-08-17,2026-08-18,2026-08-19,2026-08-20,2026-08-21
CodigoAlmacen,,,,,,,,,,,,,,,,,,,,,
GT1B2B,151,151,150,150,156,153,153,154,154,155,...,164,162,163,163,163,163,164,167,167,187
GT1CEDI,2061,2061,2049,2045,2046,2039,2044,2037,2037,2025,...,2032,2023,2025,0,0,0,2005,1995,2010,2312
GT1COMER,18552,17946,17972,18001,18138,18146,18212,18170,18093,18028,...,18282,18306,18368,18319,18244,18242,18275,18283,18213,18193
GT1CONTI,247,247,247,247,247,247,247,247,247,247,...,247,247,247,247,247,247,247,247,247,247
GT1DAÑO,260,260,262,266,266,266,408,409,409,415,...,449,449,449,449,449,449,451,451,462,463
GT1RECL,89,89,90,90,90,90,90,90,90,90,...,87,87,87,87,87,87,87,87,87,88
GT1SERV,1926,1926,1927,1938,1938,1941,1892,1890,1890,1909,...,1929,1914,1918,1918,1918,1932,1959,1959,1959,1957
GT1TRAN,370,784,780,809,803,808,680,664,664,672,...,628,641,632,632,632,645,652,637,631,190
GT2CEDI,155,155,151,157,152,150,156,152,152,147,...,157,158,156,156,156,152,147,146,146,144


Inventory dates cover August 1–21, 2026. GT1CEDI has no observations on August 15–17. GT2COMER appears only on August 20, with one row. These gaps require investigation and will not be interpreted as zero stock.

In [58]:
inventario_diario.duplicated().sum()

np.int64(0)

In [59]:
inventory_key = ["DataMartDate", "pais", "CodigoAlmacen", "Ubicacion", "SKU"] # Adding GrupoArticulo removes duplicate keys completely
inventario_diario.duplicated(inventory_key).sum() 

np.int64(246)

In [60]:
duplicate_inventory = inventario_diario.loc[inventario_diario.duplicated(inventory_key, keep=False)]
duplicate_inventory.sort_values(inventory_key).head(20)

,DataMartDate,pais,Canal,Ubicacion,CodigoAlmacen,GrupoArticulo,SKU,sum_Stock,sum_Comprometido,sum_Pedido,sum_CostoTotal,sum_CostoTotalUSD
564,2026-08-01,SLC-GT,DISTY,GT1CEDI-GT-001-UBI-FIS1- DISTY,GT1CEDI,PROMOCIONALES,EO-IA500BBEGWW,5,0,0,0.050000,0.01
590,2026-08-01,SLC-GT,DISTY,GT1CEDI-GT-001-UBI-FIS1- DISTY,GT1CEDI,TELEFONIA Y ACC,EO-IA500BBEGWW,151,0,0,7281.971980,955.03
1770,2026-08-01,SLC-GT,RETAIL,GT1CEDI-GT-001-UBI-FIS1-.COM,GT1CEDI,PROMOCIONALES,EF-VF731PAEGWW,4,0,0,0.040000,0.01
2112,2026-08-01,SLC-GT,RETAIL,GT1CEDI-GT-001-UBI-FIS1-.COM,GT1CEDI,TELEFONIA Y ACC,EF-VF731PAEGWW,2,0,0,722.060000,94.70
3662,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-502-UBI-RETAIL,GT1COMER,PROMOCIONALES,EO-IA500BBEGWW,1,0,0,0.010000,0.00
3834,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-502-UBI-RETAIL,GT1COMER,TELEFONIA Y ACC,EO-IA500BBEGWW,13,0,0,573.269152,75.18
4492,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-504-UBI-RETAIL,GT1COMER,PROMOCIONALES,EO-IA500BBEGWW,3,0,0,0.030000,0.00
4652,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-504-UBI-RETAIL,GT1COMER,TELEFONIA Y ACC,EO-IA500BBEGWW,10,0,0,477.715960,62.65
5670,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-507-UBI-RETAIL,GT1COMER,PROMOCIONALES,EO-IA500BBEGWW,1,0,0,0.010000,0.00
5798,2026-08-01,SLC-GT,RETAIL,GT1COMER-GT-507-UBI-RETAIL,GT1COMER,TELEFONIA Y ACC,EO-IA500BBEGWW,6,0,0,191.106384,25.06


No completely identical inventory rows were found. However, the candidate key (date, country, warehouse, location, SKU) has 246 repeated occurrences beyond the first.

The displayed examples differ in GrupoArticulo, stock and cost, including PROMOCIONALES versus TELEFONIA Y ACC. This suggests the proposed key may omit part of the inventory grain. These rows will not be removed until that distinction is understood. Adding GrupoArticulo to the categorykey removes the duplicates meaning we can assume that these a couple promotional items and the remainder is full price.

In [61]:
duplicate_inventory.value_counts("GrupoArticulo")

GrupoArticulo
TELEFONIA Y ACC    246
PROMOCIONALES      225
REPUESTOS           21
Name: count, dtype: int64

In [62]:
inventory_measures = [
    "sum_Stock",
    "sum_Comprometido",
    "sum_Pedido",
    "sum_CostoTotal",
]
inventario_diario[inventory_measures].describe()


,sum_Stock,sum_Comprometido,sum_Pedido,sum_CostoTotal
count,502838.000000,502838.000000,502838.000000,5.023910e+05
mean,11.053319,0.247976,0.102091,1.053963e+04
std,174.878688,39.748253,7.013489,1.741225e+05
min,1.000000,0.000000,0.000000,0.000000e+00
25%,1.000000,0.000000,0.000000,1.661344e+02
50%,2.000000,0.000000,0.000000,5.329099e+02
75%,3.000000,0.000000,0.000000,3.540896e+03
max,37104.000000,17290.000000,3200.000000,3.820124e+07


In [63]:
inventario_diario[inventory_measures].lt(0).sum()

sum_Stock           0
sum_Comprometido    0
sum_Pedido          0
sum_CostoTotal      0
dtype: int64

No negative values were found in stock, committed quantity, ordered quantity or recorded GTQ cost. The minimum observed stock is 1: the supplied rows contain no explicit zero-stock observations. Missing observations therefore require special care when evaluating stock availability.

In [64]:
etiquetas.info()
display(etiquetas.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_combo        916 non-null    object
 1   SKU             916 non-null    object
 2   CodigoAlmacen   916 non-null    object
 3   riesgo_quiebre  916 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 28.8+ KB


,id_combo,SKU,CodigoAlmacen,riesgo_quiebre
0,C0001,0201-004132,GT1SERV,0
1,C0005,13509-132,GT1SERV,0
2,C0006,2501-001011,GT1SERV,0
3,C0007,2501-001035,GT1SERV,0
4,C0008,3001-002871,GT1SERV,0


In [65]:
combos.info()
display(combos.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_combo       392 non-null    object
 1   SKU            392 non-null    object
 2   CodigoAlmacen  392 non-null    object
dtypes: object(3)
memory usage: 9.3+ KB


,id_combo,SKU,CodigoAlmacen
0,C0002,12555601,GT1COMER
1,C0003,13509-132,GT1CEDI
2,C0004,13509-132,GT1COMER
3,C0012,AGE83X/XAP,GT2CEDI
4,C0013,AH94-00022W,GT1SERV


In [66]:
for name, table in [("etiquetas", etiquetas), ("combos", combos)]:
    print(name, table.shape)
    display(table.isna().sum())
    print("Exact duplicate rows:", table.duplicated().sum())

etiquetas (916, 4)


id_combo          0
SKU               0
CodigoAlmacen     0
riesgo_quiebre    0
dtype: int64

Exact duplicate rows: 0
combos (392, 3)


id_combo         0
SKU              0
CodigoAlmacen    0
dtype: int64

Exact duplicate rows: 0


In [67]:
for name, table in [("etiquetas", etiquetas), ("combos", combos)]:
    print(name, "duplicate IDs:", table.duplicated("id_combo").sum())

etiquetas duplicate IDs: 0
combos duplicate IDs: 0


In [68]:
etiquetas["riesgo_quiebre"].value_counts(dropna=False)

riesgo_quiebre
0    841
1     75
Name: count, dtype: int64

In [69]:
etiquetas["id_combo"].isin(combos["id_combo"]).sum()

np.int64(0)

etiquetas contains 916 labeled combinations and combos contains 392 combinations to predict. Neither file has missing values, exact duplicate rows or duplicate id_combo values. No IDs overlap between the files.

The target contains 841 negative cases and 75 positive cases (approximately 8.2% positive), so accuracy alone would be insufficient to evaluate the model. Uniqueness and overlap of SKU-warehouse pairs, and their inventory coverage at the cutoff, remain to be checked.